In [78]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torchvision.transforms as transforms
from torchmetrics.classification import Accuracy
from torchvision.datasets import Imagenette, SVHN
from torchvision import models
from pytorch_lightning.callbacks import ModelCheckpoint

torch.set_float32_matmul_precision('high')

In [79]:
k = 2  # Number of workers

In [80]:
# Define the ResNet model in PyTorch Lightning
class ResNetPLModel(pl.LightningModule):
    def __init__(self, num_classes=1000, resnet_version='resnet50'):
        super(ResNetPLModel, self).__init__()

        # Load the ResNet model
        self.model = {
            'resnet50': models.resnet50, 'resnet18': models.resnet18
        }[resnet_version](weights='DEFAULT')

        self.loss_fn = nn.CrossEntropyLoss()
        self.accuracy = Accuracy(
            task='multiclass', num_classes=num_classes, average='weighted')

    def forward(self, x):
        return self.model(x)

    def step_with_custom_logs(self, stage, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        # acc = self.accuracy(logits, y)

        # Log metrics
        self.log(f'{stage}_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        # self.log(f'{stage}_acc', acc, on_step=True, on_epoch=True, prog_bar=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self.step_with_custom_logs('train', batch, batch_idx)

    def validation_step(self, batch, batch_idx):
        return self.step_with_custom_logs('valid', batch, batch_idx)

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9)
        return optimizer

    def clone(self):
        new_model = ResNetPLModel(num_classes=self.accuracy.num_classes,
                                  resnet_version='resnet18')
        new_model.load_state_dict(self.state_dict())
        return new_model

In [81]:
dataset = [
    Imagenette(
        root='./data/Imagenet', split=s,
        transform=transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    ) for s in ['train', 'val']]

dataset = [
    SVHN(
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    ) for s in ['train', 'test']]

In [82]:
dataset = [torch.utils.data.Subset(d, list(range(200))) for d in dataset]

In [83]:
def pre_send_process(single_model_grad_agent):
    return single_model_grad_agent

def server_rec_process(all_encoded_model_grad):
    return all_encoded_model_grad

In [84]:
import importlib
import FL_sim

importlib.reload(FL_sim)
from FL_sim import FLSimulator

#num_agents: int, communication_rounds: int, client_epochs_per_round: int,
                 # batch_size: int, dataset_train: VisionDataset, dataset_test: VisionDataset,
                 # pl_model: pl.LightningModule, aggregation_method='fedavg',
                 # iid_data=False, pre_send_process=None,
                 # server_rec_process:Optional[Callable[[List[Dict]], List[Dict]]]=None)
sim = FLSimulator(
    num_agents=k, communication_rounds=5, client_epochs_per_round=1,
    batch_size=64, dataset_train=dataset[0], dataset_test=dataset[1],
    pl_model=ResNetPLModel(num_classes=10, resnet_version='resnet18'),
    aggregation_method='fedavg', iid_data=True,
    pre_send_process=pre_send_process, server_rec_process=server_rec_process
)

In [86]:
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

sim.run_simulation()

round 1/5
loss: 5.068951606750488


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x000001FC3FF97C40>
Traceback (most recent call last):
  File "D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\torch\utils\data\dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\torch\utils\data\dataloader.py", line 1438, in _shutdown_workers
    self._mark_worker_as_unavailable(worker_id, shutdown=True)
  File "D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\torch\utils\data\dataloader.py", line 1380, in _mark_worker_as_unavailable
    assert self._workers_status[worker_id] or (self._persistent_workers and shutdown)
           ^^^^^^^^^^^^^^^^^^^^
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x000001FC3FF97C40>
Traceback (most recent call last):
  File "D:\User\Software\Mi

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "D:\User\Software\MiniConda3\envs\SuperCondaEnv\Lib\site-packages\IPython\core\interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\hosse\AppData\Local\Temp\ipykernel_27288\2749627013.py", line 4, in <module>
    sim.run_simulation()
  File "D:\User\App Files\Projects\VUB-ACS-25_Thesis\FL_sim.py", line 147, in run_simulation
    self._aggregate_models()
  File "D:\User\App Files\Projects\VUB-ACS-25_Thesis\FL_sim.py", line 124, in _aggregate_models
    self.server_rec_process([agent.get_accum_grads() for agent in self.agents]),
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\User\App Files\Projects\VUB-ACS-25_Thesis\FL_sim.py", line 124, in <listcomp>
    self.server_rec_process([agent.get_accum_grads() for agent in self.agents]),
                             ^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\User\App Files\Projects\VUB-ACS-25_Thesis\FL_sim.

In [47]:
import importlib
import Componenets

importlib.reload(Componenets)
from Componenets import grad_quantizer, grad_encoder, grad_decoder, grad_de_quantizer


def sim_worker(model, train_loader):
    model.load_state_dict(server_model_to_send)
    trainer = pl.Trainer(max_epochs=5, enable_progress_bar=False)
    trainer.fit(model, train_loader)

    quant = grad_quantizer(model.latest_parameters)
    encode = grad_encoder(quant)
    return quant, encode


def sim_server(quant_list, encode_list):
    decoded = grad_decoder(worker_encode)
    de_quant = grad_de_quantizer(decoded, temp)
    return de_quant

In [40]:
signals_data = {'raw': [], 'quant': [], 'encode': []}

# worker side
for i in range(len(train_loaders)):
    worker_quant, worker_encode = sim_worker(worker_models_list[i], train_loaders[i])
    signals_data['quant'].append(worker_quant)
    signals_data['encode'].append(worker_encode)
    signals_data['raw'].append(worker_models_list[i].latest_parameters)

# server side
s_decoded = grad_decoder(signals_data['encode'])
temp = [[a, b.shape] for a, b in worker_models_list[0].latest_parameters.copy()]
s_de_quant = grad_de_quantizer(s_decoded, temp)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params
---------------------------------------------
0 | model   | Sequential       | 4.3 M 
1 | loss_fn | CrossEntropyLoss | 0     
---------------------------------------------
4.3 M     Trainable params
0         Non-trainable params
4.3 M     Total params
17.081    Total estimated model params size (MB)
`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params
---------------------------------------------
0 | model   | Sequential       | 4.3 M 
1 | loss_fn | CrossEntropyLoss | 0     
---------------------------------------------
4.3 M    

NameError: name 'server_agg' is not defined

In [49]:
# aggregate
worker_count = len(train_loaders)
final_grad = [[s_de_quant[0][i][0], s_de_quant[0][i][1] / worker_count] for i in range(len(s_de_quant[0]))]
for i in range(1, len(s_de_quant)):
    for j in range(len(final_grad)):
        final_grad[j][1] += s_de_quant[i][j][1] / worker_count

In [50]:
# update server model via optimizer
optimizer = model.configure_optimizers()
optimizer.zero_grad()
for i in range(len(final_grad)):
    name, grad = final_grad[i]
    param = dict(model.named_parameters())['model.' + name]
    param.grad = torch.tensor(grad).to(param.device)
optimizer.step()